<br>

# Geoserver

<br>

Michel Metran\
Data: 21.06.2025\
Atualizado em: 21.06.2025


In [ ]:
import urllib
from pathlib import Path
from urllib.parse import (
    parse_qs,
    parse_qsl,
    quote,
    quote_plus,
    unquote,
    urlencode,
    urljoin,
    urlparse,
    urlsplit,
    urlunparse,
)
from urllib.request import urlopen, urlretrieve

import geopandas as gpd
from owslib.csw import CatalogueServiceWeb
from owslib.wcs import WebCoverageService
from owslib.wfs import WebFeatureService
from owslib.wms import WebMapService

<br>

---

## URL


In [ ]:
url = 'http://datageo.ambiente.sp.gov.br/geoserver/ows'
url = 'http://datageo.ambiente.sp.gov.br/geoserver/web'

In [ ]:
# aaa = urlparse(url=url)
# aaa

In [ ]:
# ows_path = (Path(aaa.path).parent / 'ows').as_posix()
# ows_path

In [ ]:
# aaa._replace(path=ows_path)

In [ ]:
# urlunparse(aaa)

In [ ]:
class Geoserver:
    def __init__(self, url):
        """
        Initializes the Geoserver instance.
        This method checks if the provided URL ends with 'web' and modifies it to point to the OWS path.

        :param url: Endereço do Geoserver, que deve terminar com 'web'.
        :type url: str
        """
        url_parsed = urlparse(url=url)
        print(url_parsed)
        end = Path(url_parsed.path).parts[-1]
        if end != 'web':
            raise ValueError(f'Invalid URL: {url}. Expected to end with "web".')

        # Altera para o caminho correto do OWS
        # url_parsed = urlparse(url=url)
        ows_path = (Path(url_parsed.path).parent / 'ows').as_posix()
        url = urlunparse(components=url_parsed._replace(path=ows_path))

        self.url = url
        self.wms = WebMapService(url)

        # Observei que o "path" do DataGeo, para obter dados, é diferente de "/geoserver/web"
        # É, na real, "/geoserver/web"
        url_parsed = urlparse(url=url)
        print(url_parsed)
        if url_parsed.netloc == 'datageo.ambiente.sp.gov.br':
            url = urlunparse(
                components=url_parsed._replace(path='/geoserver/datageo/ows/')
            )
            self.url = url

        # self.wfs = WebFeatureService(url)
        # self.wcs = WebCoverageService(url)
        # self.csw = CatalogueServiceWeb(url)

    @property
    def list_wms_layers(self):
        list_layers = self.wms.contents.keys()
        # for layer_name, layer in self.wms.contents.items():
        #     if layer.queryable == 1:
        #         print(f"Layer: {layer_name}")
        # return self.wms.contents
        list_layers = list(list_layers)
        list_layers.sort()
        return list_layers

    def create_url(
        self, layer='datageo:AEROPORTOS_PUBLICOS_ANAC_2021', format='GeoJSON'
    ):

        if layer not in self.list_wms_layers:
            raise ValueError(f'Layer {layer} not found in WMS layers.')

        if format not in ['GeoJSON', 'GML2', 'GML3', 'Shapefile']:
            raise ValueError(
                f'Format {format} not supported. Use GeoJSON, GML2, or GML3.'
            )

        dd_type = {
            'GeoJSON': 'application/json',
            'GML2': 'application/gml+xml; version=2',
            'GML3': 'application/gml+xml; version=3',
            'Shapefile': 'SHAPE-ZIP',
        }

        dd = {
            # service': 'WMS',
            'service': 'WFS',
            'version': '1.0.0',
            'request': 'GetFeature',
            'typeName': layer,
            'maxFeatures': '500',
            'outputFormat': dd_type[format],
        }
        params = urlencode(query=dd)
        return urlunparse(urlparse(url=gs.url)._replace(query=params))

    # def get_wms_capabilities(self):
    #     return self.wms.getcapabilities()

    # def get_wfs_capabilities(self):
    #     return self.wfs.getcapabilities()

    # def get_wcs_capabilities(self):
    #     return self.wcs.getcapabilities()

    # def get_csw_capabilities(self):
    #     return self.csw.getcapabilities()


gs = Geoserver(url=url)
gs.url

In [ ]:
bbb = gs.create_url(
    layer='datageo:AEROPORTOS_PUBLICOS_ANAC_2021',
    # format='GeoJSON'
    format='Shapefile',
)
bbb

In [ ]:
gdf = gpd.read_file(filename=bbb)
gdf

In [ ]:
gs.list_wms_layers

In [ ]:
dd = {
    # service': 'WMS',
    'service': 'WFS',
    'version': '1.0.0',
    'request': 'GetFeature',
    'typeName': 'datageo:AEROPORTOS_PUBLICOS_ANAC_2021',
    'maxFeatures': '50',
    'outputFormat': 'application/json',
}

In [ ]:
dd_type = {
    'GeoJSON': 'application/json',
    'GML2': 'application/gml+xml; version=2',
    'GML3': 'application/gml+xml; version=3',
}

dd_type['GeoJSON']

In [ ]:
urlencode(dd)

In [ ]:
'service=WFS&version=1.0.0&request=GetFeature&typeName=datageo:AEROPORTOS_PUBLICOS_ANAC_2021&maxFeatures=50&outputFormat=application%2Fjson'

In [ ]:
url = 'http://datageo.ambiente.sp.gov.br/geoserver/ows?service=WFS&version=1.1.0&request=GetCapabilities'
url = 'https://datageo.ambiente.sp.gov.br/geoserver/datageo/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=datageo:00_AGLOMERACOES_URBANAS_2018_POL&maxFeatures=50&outputFormat=application%2Fjson'

In [ ]:
urlparse(url=url)
# url = urlunparse(components=urlparse(url=url)._replace(query=urlencode(dd)))
# url
# ParseResult(scheme='http', netloc='datageo.ambiente.sp.gov.br', path='/geoserver/web', params='', query='', fragment='')

In [ ]:
urlsplit(url=url).path
url_parsed = urlparse(url=url)
url_parsed

In [ ]:
url = urlunparse(components=url_parsed._replace(path='/geoserver/datageo/ows'))
url

In [ ]:
dict(parse_qsl(qs=urlsplit(url=url).query))

<br>

---

## WFS


In [ ]:
# url = 'http://datageo.ambiente.sp.gov.br/geoserver/ows?service=WMS&version=1.3.0&request=GetCapabilities'
url = 'http://datageo.ambiente.sp.gov.br/geoserver/ows'
url = 'http://datageo.ambiente.sp.gov.br/geoserver/datageo/ows'
wfs = WebFeatureService(url=url, version='2.0.0')

In [ ]:
# Lista os tipos de feições disponíveis
for feature in list(wfs.contents):
    print(feature)

In [ ]:
[op.name for op in wms.operations]

In [ ]:
wfs.identification.title

In [ ]:
list(wfs.contents)

In [ ]:
wfs.getOperationByName('GetFeature').name
wfs.getOperationByName('GetFeature').formatOptions
wfs.getOperationByName('GetFeature').parameters
wfs.getOperationByName('GetFeature').methods
wfs.getOperationByName('GetFeature').constraints

In [ ]:
# Obter os dados no formato GeoJSON (ou outro formato suportado)
response = wfs.getfeature(
    typename='datageo:VWM_BIOMA_BRASIL_5000000_IBGE_POL',
    # bbox=(173700, 440400, 178700, 441400),
    # srsname='EPSG:28992'
    # srsname='EPSG:4326',
    srsname='EPSG:4674',
    outputFormat='application/json',
)
response

In [ ]:
gdf = gpd.read_file(filename=bbb)
gdf.crs

In [ ]:
gdf.info()

In [ ]:
wfs.getcapabilities().read()

In [ ]:
# Salvar os dados em um arquivo
with open("dados.geojson", "wb") as f:
    f.write(response.read())

<br>

---

## Geoserver


In [ ]:
# url = 'http://datageo.ambiente.sp.gov.br/geoserver/ows?service=WMS&version=1.3.0&request=GetCapabilities'
url = 'http://datageo.ambiente.sp.gov.br/geoserver/ows'
wms = WebMapService(url=url)

In [ ]:
# url = 'http://datageo.ambiente.sp.gov.br/geoserver/ows?service=WMS&version=1.3.0&request=GetCapabilities'
url = 'http://datageo.ambiente.sp.gov.br/geoserver/ows'
wms = WebMapService(url=url)

for layer in wms.contents:
    # print(layer)
    pass

for layer_name, layer in wms.contents.items():
    if layer.queryable == 1:
        print(f"Layer: {layer_name}")
        print(f"  Title: {layer.title}")
        print(f"  Abstract: {layer.abstract}")
        print(f"  BoundingBox: {layer.boundingBoxWGS84}")
        print(f"  CRS: {layer.crsOptions}")
        print(f"  Styles: {layer.styles}")
        print(f"  Keywords: {layer.keywords}")
        print(f"  Queryable: {layer.queryable}")
        print(f"  Opaque: {layer.opaque}")
        # print(f"  Dimensions: {layer.dimensions}")
        print(f"  MetadataURLs: {layer.metadataUrls}")
        print()

In [ ]:
urlsplit